# DS211 Lab 2 — Kaggle Orchestrator

Notebook này chỉ điều phối: config, EDA/report, progress, train tuần tự và evaluation sau cùng. Logic nằm trong `src/`.

In [ ]:
# Run after clone/pull and `pip install -q -r requirements.txt`.
from pathlib import Path
import gc, subprocess, sys, yaml
import torch

REPO = Path('/kaggle/working/DS201_Lab2')
SOURCE_ROOT = Path('/kaggle/input/datasets/mphatfromuit/vinafood21/VinaFood21')
if not (SOURCE_ROOT / 'train').is_dir() or not (SOURCE_ROOT / 'test').is_dir():
    raise FileNotFoundError(f'Incorrect SOURCE_ROOT: {SOURCE_ROOT}. Run !find /kaggle/input -maxdepth 3 -type d | head -30')

# Create editable per-session YAMLs from Git-tracked templates.
RUNTIME_DIR = REPO / 'configs/runtime'
WORKING_DATA = Path('/kaggle/working/data')
REPORT_ROOT = Path('/kaggle/working/report')
SPECS = {
    'lenet_28_t4.yaml': ('lenet_28.example.yaml', 'lenet_28_seed42'),
    'googlenet_64_t4.yaml': ('googlenet_64.example.yaml', 'googlenet_64_seed42'),
    'resnet18_96_t4.yaml': ('local.example.yaml', 'resnet18_96_seed42'),
    'resnet50_224_t4.yaml': ('resnet50_224.example.yaml', 'resnet50_224_seed42'),
}
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for runtime_name, (template_name, run_name) in SPECS.items():
    runtime_path = RUNTIME_DIR / runtime_name
    if not runtime_path.exists():
        cfg = yaml.safe_load((REPO / 'configs' / template_name).read_text(encoding='utf-8'))
        size = cfg['data']['image_size']
        cfg['paths'].update({
            'source_root': str(SOURCE_ROOT),
            'split_manifest': str(WORKING_DATA / 'metadata/split_seed42.csv'),
            'resized_root': str(WORKING_DATA / f'resized_{size}'),
            'resized_manifest': str(WORKING_DATA / f'resized_{size}/manifest.csv'),
            'class_mapping': str(WORKING_DATA / 'metadata/class_to_idx.json'),
            'report_root': str(REPORT_ROOT),
            'run_dir': str(Path('/kaggle/working/runs') / run_name),
        })
        runtime_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

# Required tasks first; append optional Bài 3*/4* only when desired.
CONFIGS = [
    RUNTIME_DIR / 'lenet_28_t4.yaml',
    RUNTIME_DIR / 'googlenet_64_t4.yaml',
    # RUNTIME_DIR / 'resnet18_96_t4.yaml',
    # RUNTIME_DIR / 'resnet50_224_t4.yaml',
]

def run_script(*arguments: str) -> None:
    subprocess.run([sys.executable, *arguments], cwd=REPO, check=True)


In [ ]:
# Phase 1 — create exactly one frozen split, then materialize and smoke-test each config.
run_script('scripts/prepare.py', '--config', str(CONFIGS[0]), '--create-split')

for config_path in CONFIGS:
    run_script('scripts/prepare.py', '--config', str(config_path), '--materialize')
    run_script('scripts/smoke_test.py', '--config', str(config_path), '--use-materialized')


In [ ]:
# Phase 2 — train sequentially. Run only when Phase 1 has passed.
for config_path in CONFIGS:
    run_script('scripts/train_config.py', '--config', str(config_path))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Phase 3 — only after every config in Phase 2 finishes, evaluate best checkpoints sequentially.
REPORT_ROOT = Path('/kaggle/working/report')
for config_path in CONFIGS:
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    run_dir = Path(config['paths']['run_dir'])
    output_dir = REPORT_ROOT / '03_evaluation' / run_dir.name
    run_script('scripts/evaluate_checkpoint.py', '--config', str(config_path),
               '--checkpoint', str(run_dir / 'best.pt'), '--output-dir', str(output_dir))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

run_script('scripts/package_report.py', '--report-dir', str(REPORT_ROOT),
           '--zip-path', '/kaggle/working/report.zip')
